# AIC 2026 Kaggle pilot

Pilot end-to-end cho 3–10 video: ingest → SigLIP2/FAISS → ASR/OCR Chronicle → text indexes → FastAPI UI.

Trước khi chạy: attach Kaggle Dataset chứa video, chọn GPU, bật Internet để tải package/model. Không cần Kaggle API token, Gemini key, ngrok token hoặc Cloudflare token cho cấu hình mặc định.

In [ ]:
import sys
from pathlib import Path

# BẮT BUỘC: thay đúng đường dẫn dataset đã attach. Không tự dò vì một
# notebook có thể attach nhiều dataset.
KAGGLE_VIDEO_DIR = Path("/kaggle/input/REPLACE_WITH_YOUR_VIDEO_DATASET")

# Repository public chứa pipeline.
REPO_URL = "https://github.com/REPLACE_WITH_OWNER/AIC2026_prepare-main.git"
REPO_DIR = Path("/kaggle/working/AIC2026_prepare-main")
DATA_ROOT = Path("/kaggle/working/data")

# Optional: artifact dataset của session trước, ví dụ
# /kaggle/input/aic2026-pilot-artifacts/data. Để None khi chạy mới.
PREVIOUS_DATA_ROOT = None

# Dataset lớn hơn vẫn dùng được: notebook liên kết 10 video đầu theo thứ tự
# đường dẫn. Muốn chọn video khác, trỏ KAGGLE_VIDEO_DIR vào subfolder pilot.
PILOT_VIDEO_LIMIT = 10
SMOKE_VIDEO_LIMIT = 3
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm"}
MANUAL_QUERY = "mô tả đúng một cảnh có trong video pilot"

assert "REPLACE_WITH" not in str(KAGGLE_VIDEO_DIR), "Hãy đặt KAGGLE_VIDEO_DIR"
assert "REPLACE_WITH" not in REPO_URL, "Hãy đặt REPO_URL"

In [ ]:
import platform
import shutil
import socket
import subprocess

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working disk:", shutil.disk_usage("/kaggle/working"))
subprocess.run(["nvidia-smi"], check=False)
try:
    print("Internet DNS:", socket.gethostbyname("huggingface.co"))
except OSError as exc:
    raise RuntimeError("Bật Internet trong Kaggle Settings trước khi cài/tải model") from exc

In [ ]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Reuse existing repo: {REPO_DIR}")
CONFIG = REPO_DIR / "configs/t0-kaggle.yaml"

# Giữ lỗi dependency rõ ràng. Không cài requirements-gpu.txt và không tự
# thay faiss-cpu bằng faiss-gpu trong pilot.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Git commit:", commit)

> Nếu pip báo cần restart kernel vì thay NumPy/Torch hoặc import vẫn dùng phiên bản cũ, restart session rồi chạy lại từ cell Settings. Không bỏ qua cảnh báo binary incompatibility.

In [ ]:
import json
import shutil

ARTIFACT_DIRS = ("keyframes", "manifests", "embeddings", "indexes", "reports")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if PREVIOUS_DATA_ROOT is not None:
    previous = Path(PREVIOUS_DATA_ROOT)
    if not previous.is_dir():
        raise FileNotFoundError(f"Artifact root không tồn tại: {previous}")
    for name in ARTIFACT_DIRS:
        source = previous / name
        target = DATA_ROOT / name
        if source.exists() and not target.exists():
            shutil.copytree(source, target)
            print(f"Restored {name}")

if not KAGGLE_VIDEO_DIR.is_dir():
    raise FileNotFoundError(f"Kaggle video directory không tồn tại: {KAGGLE_VIDEO_DIR}")
all_videos = sorted(
    p
    for p in KAGGLE_VIDEO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
)
if len(all_videos) < SMOKE_VIDEO_LIMIT:
    raise ValueError(
        f"Smoke pilot cần ít nhất {SMOKE_VIDEO_LIMIT} video; tìm thấy {len(all_videos)}."
    )
videos = all_videos[:PILOT_VIDEO_LIMIT]
if len(all_videos) > PILOT_VIDEO_LIMIT:
    print(
        f"Found {len(all_videos)} videos; link only the first {PILOT_VIDEO_LIMIT}. "
        "Point KAGGLE_VIDEO_DIR at a pilot subfolder to choose another subset."
    )

videos_dir = DATA_ROOT / "videos"
videos_dir.mkdir(parents=True, exist_ok=True)
for old in videos_dir.iterdir():
    if old.is_symlink():
        old.unlink()
    else:
        raise RuntimeError(f"Refuse xóa file không phải symlink: {old}")

stems = set()
for source in videos:
    if source.stem in stems:
        raise ValueError(f"Trùng video_id (file stem): {source.stem}")
    stems.add(source.stem)
    (videos_dir / source.name).symlink_to(source)
print(f"Linked {len(videos)} videos into {videos_dir}")

manifest = DATA_ROOT / "manifests/videos.jsonl"
if manifest.exists():
    missing = []
    for line in manifest.read_text(encoding="utf-8").splitlines():
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if row.get("path") and not Path(row["path"]).exists():
            missing.append(row["path"])
    if missing:
        raise RuntimeError(
            f"Manifest cũ trỏ tới {len(missing)} path không còn tồn tại; "
            "giữ cùng Kaggle dataset slug hoặc chạy ingest sạch."
        )

In [ ]:
import torch

from aic.config import load_config

cfg = load_config(CONFIG)
assert cfg.paths.data_root == DATA_ROOT
assert cfg.chronicle.caption.enabled is False
assert cfg.cortex.enabled is False
assert cfg.index.device == "cpu"
print("Config valid:", CONFIG)
print("CUDA available:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
print("FAISS search device:", cfg.index.device)
print("ASR model:", cfg.chronicle.asr.model_size)

In [ ]:
import hashlib
import json
import subprocess
import time
from datetime import datetime, timezone


def run_stage(name, *args):
    command = [sys.executable, str(REPO_DIR / "scripts" / name), *map(str, args)]
    print("$", " ".join(command))
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO_DIR, check=True)
    print(f"Completed in {(time.perf_counter() - started) / 60:.1f} min")


def tree_size(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) if path.exists() else 0


def print_sizes():
    for name in (*ARTIFACT_DIRS,):
        print(f"{name:12s} {tree_size(DATA_ROOT / name) / 2**30:8.3f} GiB")
    print(f"{'model-cache':12s} {tree_size('/kaggle/working/model-cache') / 2**30:8.3f} GiB")

## Nhịp A: smoke test 3 video, 1 GPU

`--limit` chỉ có ý nghĩa single-GPU. Embed xử lý toàn bộ keyframe của ba video đã ingest để index không bị thiếu có chủ ý.

In [ ]:
run_stage("ingest_corpus.py", "--config", CONFIG, "--num-gpus", 1, "--limit", SMOKE_VIDEO_LIMIT)
print_sizes()

In [ ]:
from PIL import Image

manifests = DATA_ROOT / "manifests"
for name in ("videos.jsonl", "shots.jsonl", "keyframes.jsonl"):
    path = manifests / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing {path}"

keyframe_rows = [json.loads(line) for line in (manifests / "keyframes.jsonl").read_text().splitlines() if line.strip()]
assert keyframe_rows, "Không có keyframe"
for row in keyframe_rows[:10]:
    with Image.open(row["image_path"]) as image:
        image.verify()
print(f"Validated {min(10, len(keyframe_rows))}/{len(keyframe_rows)} keyframes")

In [ ]:
run_stage("embed_corpus.py", "--config", CONFIG, "--num-gpus", 1)
print_sizes()

In [ ]:
model_slug = cfg.embed.model_id.replace("/", "--")
embed_dir = DATA_ROOT / "embeddings/keyframes" / model_slug
index_dir = DATA_ROOT / "indexes" / f"keyframes-{model_slug}"
assert (embed_dir / "manifest.jsonl").is_file()
for name in ("meta.json", "ids.json", "vectors.npy", "index.faiss"):
    assert (index_dir / name).is_file(), f"Missing {index_dir / name}"
meta = json.loads((index_dir / "meta.json").read_text())
print("Keyframe index:", meta)

In [ ]:
# Full local Chronicle: Faster-Whisper ASR + EasyOCR; caption/entity/VQA off.
run_stage("build_chronicle.py", "--config", CONFIG, "--num-gpus", 1, "--skip-entities")
print_sizes()

In [ ]:
run_stage("check_corpus.py", "--config", CONFIG)
run_stage("build_text_indexes.py", "--config", CONFIG)
print_sizes()

chronicle = DATA_ROOT / "manifests/chronicle.jsonl"
assert chronicle.is_file() and chronicle.stat().st_size > 0
text_indexes = [p.name for p in (DATA_ROOT / "indexes").iterdir() if p.name.startswith("chronicle-")]
assert text_indexes, "Không có text index; kiểm tra ASR/OCR coverage"
print("Text indexes:", text_indexes)

In [ ]:
import httpx

server_log = open("/kaggle/working/aic-server.log", "w")
server = subprocess.Popen(
    [sys.executable, str(REPO_DIR / "scripts/serve.py"), "--config", str(CONFIG)],
    cwd=REPO_DIR, stdout=server_log, stderr=subprocess.STDOUT,
)
for _ in range(120):
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError(Path("/kaggle/working/aic-server.log").read_text()[-4000:])
    try:
        response = httpx.post("http://127.0.0.1:8000/api/search", json={"query": MANUAL_QUERY, "top_k": 5}, timeout=10)
        if response.status_code == 200:
            break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise TimeoutError("FastAPI không sẵn sàng sau 120 giây")

payload = response.json()
assert payload.get("results"), payload
first = payload["results"][0]
assert first.get("video_id") and first.get("timestamp_ms") is not None
print(json.dumps({k: first.get(k) for k in ("video_id", "timestamp_ms", "score", "frames")}, ensure_ascii=False, indent=2))
if first.get("frames"):
    frame = httpx.get("http://127.0.0.1:8000" + first["frames"][0], timeout=10)
    assert frame.status_code == 200 and frame.headers.get("content-type", "").startswith("image/")
    print("Frame endpoint OK:", len(frame.content), "bytes")

## Optional: Cloudflare Quick Tunnel

Chỉ chạy sau khi internal API smoke test thành công. URL là **public** và UI hiện không có authentication. Không dùng tunnel để phân phối raw video/corpus. Quick Tunnel không cần token nhưng chỉ dành cho thử nghiệm; xem giới hạn hiện hành trong tài liệu Cloudflare trước khi dùng.

In [ ]:
import re
import urllib.request

cloudflared = Path("/kaggle/working/cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    cloudflared.chmod(0o755)
tunnel_log = open("/kaggle/working/cloudflared.log", "w")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT, text=True,
)
url = None
for _ in range(60):
    time.sleep(1)
    tunnel_log.flush()
    text = Path("/kaggle/working/cloudflared.log").read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError(text[-4000:])
assert url, "Không lấy được Quick Tunnel URL"
print("PUBLIC OPERATOR UI:", url)

In [ ]:
inventory = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "git_commit": commit,
    "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest(),
    "video_source": str(KAGGLE_VIDEO_DIR),
    "video_ids": sorted(stems),
    "model_ids": {"vision": cfg.embed.model_id, "text": cfg.textstack.model_id, "asr": cfg.chronicle.asr.model_size},
    "artifact_bytes": {name: tree_size(DATA_ROOT / name) for name in ARTIFACT_DIRS},
}
(DATA_ROOT / "reports").mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "reports/kaggle-inventory.json").write_text(json.dumps(inventory, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(inventory, ensure_ascii=False, indent=2))
print_sizes()

In [ ]:
# Chỉ đổi thành True sau khi đã thử UI/tunnel xong và ngay trước khi
# Save Version/tạo artifact Dataset. Thao tác này giữ artifacts nhưng loại model
# cache tải lại được và symlink video input khỏi notebook output.
PREPARE_OUTPUT_FOR_SAVE = False

if PREPARE_OUTPUT_FOR_SAVE:
    for process_name in ("tunnel", "server"):
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            process.terminate()
    videos_dir = DATA_ROOT / "videos"
    if videos_dir.exists():
        for path in videos_dir.iterdir():
            if path.is_symlink():
                path.unlink()
    model_cache = Path("/kaggle/working/model-cache")
    if model_cache.exists():
        shutil.rmtree(model_cache)
    print("Output prepared: artifacts kept; video symlinks and model cache removed.")

## Persist và chạy tiếp

Lưu `data/keyframes`, `data/manifests`, `data/embeddings`, `data/indexes`, `data/reports` thành Kaggle Dataset/Notebook Output. Không lưu raw videos hoặc `model-cache`. Session mới attach cả video dataset và artifact dataset, đặt `PREVIOUS_DATA_ROOT`, rồi chạy lại: manifest sẽ skip phần đã hoàn thành. Sau session bị kill giữa write, tạo bản config tạm bật `verify.on_resume: true`, chạy repair/resume một lần rồi tắt lại.

Nhịp B: chuẩn bị đúng 10 video trong dataset pilot, bỏ `--limit`, thử `--num-gpus 2`. Nếu OOM, quay về 1 GPU hoặc giảm batch size.